# CNN Weight Repair on Fashion-MNIST

Can a language model improve a trained image classifier by changing six classifier biases directly? This PyTorch notebook compares the original CNN, Adam fine-tuning of the full classifier head, a fixed ChatGPT bias patch, and 30 random bias patches with matching edit magnitudes.

| Section | What runs |
|---|---|
| 01 | Load Fashion-MNIST and split the original training data into training, adaptation, and validation sets |
| 02 | Train on clean images, save the epoch-one, best, and final checkpoints, and verify the best checkpoint reloads |
| 03 | Compare clean and blurred validation predictions |
| 04 | Fine-tune with Adam, inspect adaptation errors, and evaluate the direct patch and random controls |
| 05 | Compare methods on the official test split and download the outputs |

Use **Runtime → Run all** to reproduce the experiment. The fixed patch came from this baseline’s adaptation errors. On a substantially different baseline, treat it as a fixed-patch transfer test. The final download cell bundles the model weights, patch, and CSVs.

In [ ]:
# 01 | Setup and Fashion-MNIST data
from pathlib import Path
import csv
import hashlib
import json
import math
import random
import shutil
import statistics
import time

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST
from torchvision.transforms.functional import gaussian_blur

SEED = 42
EPOCHS = 4
BATCH_SIZE = 256
BLUR_SIGMA = 1.8
BLUR_KERNEL = 9
DATA_ROOT = Path('data')
OUTPUT_ROOT = Path('fashion_mnist_runs')
RUN_DIR = OUTPUT_ROOT / (time.strftime('%Y%m%d_%H%M%S') + f'_{time.time_ns() % 1_000_000_000:09d}')
RUN_DIR.mkdir(parents=True)
BASELINE_PATH = RUN_DIR / '02_baseline_best.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 4))

FashionMNIST.mirrors = [
    'https://fashion-mnist.s3.eu-central-1.amazonaws.com/',
    'https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/',
    *FashionMNIST.mirrors,
]
source = FashionMNIST(DATA_ROOT, train=True, download=True)
CLASS_NAMES = source.classes
split = torch.Generator().manual_seed(SEED)
train_ids, adapt_ids, val_ids = torch.randperm(len(source), generator=split).split([54_000, 2_000, 4_000])

def image_set(dataset, indices=None):
    images, labels = dataset.data, dataset.targets
    if indices is not None:
        images, labels = images[indices], labels[indices]
    return images.unsqueeze(1), labels

def loader(images, labels, shuffle=False):
    return DataLoader(TensorDataset(images, labels), batch_size=BATCH_SIZE,
                      shuffle=shuffle, generator=torch.Generator().manual_seed(SEED),
                      pin_memory=DEVICE.type == 'cuda')

def pixels(images, blurred=False):
    images = images.to(DEVICE, dtype=torch.float32, non_blocking=True).div_(255)
    if blurred:
        images = gaussian_blur(images, [BLUR_KERNEL] * 2, [BLUR_SIGMA] * 2)
    return images

train_loader = loader(*image_set(source, train_ids), shuffle=True)
adapt_loader = loader(*image_set(source, adapt_ids))
val_loader = loader(*image_set(source, val_ids))
print(f'PyTorch {torch.__version__} | device: {DEVICE}')
print(f'Train: {len(train_ids):,} | adaptation: {len(adapt_ids):,} | validation: {len(val_ids):,}')
print(f'Files will be saved to {RUN_DIR}')

In [ ]:
# 02 | Train, save and reload the baseline CNN
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 7 * 7, 24), nn.ReLU(),
        )
        self.classifier = nn.Linear(24, 10)

    def forward(self, images):
        return self.classifier(self.encoder(images))

def save_weights(model, path):
    torch.save({name: value.detach().cpu() for name, value in model.state_dict().items()}, path)

def load_weights(path=BASELINE_PATH):
    model = FashionCNN().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    return model.eval()

@torch.no_grad()
def predictions(model, batches, blurred=False):
    model.eval()
    scores, labels = [], []
    for images, targets in batches:
        scores.append(model(pixels(images, blurred)).cpu())
        labels.append(targets)
    return torch.cat(scores), torch.cat(labels)

def accuracy(scores, labels):
    return (scores.argmax(1) == labels).double().mean().item() * 100

def write_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as out:
        writer = csv.DictWriter(out, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

torch.manual_seed(SEED)
model = FashionCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train_loader = loader(*image_set(source, train_ids), shuffle=True)
history, best_accuracy = [], -1
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, correct = 0.0, 0
    for images, labels in train_loader:
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        scores = model(pixels(images))
        loss = F.cross_entropy(scores, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(labels)
        correct += (scores.detach().argmax(1) == labels).sum().item()
    val_scores, val_targets = predictions(model, val_loader)
    val_accuracy = accuracy(val_scores, val_targets)
    row = dict(epoch=epoch, train_loss=loss_sum / len(train_ids),
               train_accuracy_pct=100 * correct / len(train_ids), validation_accuracy_pct=val_accuracy)
    history.append(row)
    print(f"Epoch {epoch:02d} | loss {row['train_loss']:.4f} | train {row['train_accuracy_pct']:.2f}% | clean validation {val_accuracy:.2f}%", flush=True)
    if epoch == 1:
        save_weights(model, RUN_DIR / '01_epoch_01_sample.pt')
    if val_accuracy > best_accuracy:
        best_accuracy, best_epoch = val_accuracy, epoch
        best_logits = val_scores
        save_weights(model, BASELINE_PATH)

save_weights(model, RUN_DIR / '02_baseline_last.pt')
write_csv(RUN_DIR / '02_training_history.csv', history)
baseline = load_weights()
reloaded_scores, reloaded_targets = predictions(baseline, val_loader)
torch.testing.assert_close(reloaded_scores, best_logits)
assert accuracy(reloaded_scores, reloaded_targets) == best_accuracy
print(f'Checkpoint reload verified | best epoch {best_epoch:02d} | clean validation {best_accuracy:.2f}%')

In [ ]:
# 03 | Measure the effect of Gaussian blur
import matplotlib.pyplot as plt

clean_val_scores, val_targets = predictions(baseline, val_loader)
blur_val_scores, blur_val_targets = predictions(baseline, val_loader, blurred=True)
assert torch.equal(val_targets, blur_val_targets)

def confusion(scores, labels):
    return torch.bincount(10 * labels + scores.argmax(1), minlength=100).reshape(10, 10)

clean_cm = confusion(clean_val_scores, val_targets)
blur_cm = confusion(blur_val_scores, val_targets)
blur_rows = [dict(class_name='Overall', count=len(val_targets),
                  clean_accuracy_pct=accuracy(clean_val_scores, val_targets),
                  blurred_accuracy_pct=accuracy(blur_val_scores, val_targets))]
for index, name in enumerate(CLASS_NAMES):
    count = int(clean_cm[index].sum())
    blur_rows.append(dict(class_name=name, count=count,
                          clean_accuracy_pct=100 * int(clean_cm[index, index]) / count,
                          blurred_accuracy_pct=100 * int(blur_cm[index, index]) / count))
write_csv(RUN_DIR / '03_blur_validation.csv', blur_rows)
for row in blur_rows:
    print(f"{row['class_name']:12s} clean {row['clean_accuracy_pct']:6.2f}% | blurred {row['blurred_accuracy_pct']:6.2f}%")

example_images, example_labels = next(iter(val_loader))
changed = (clean_val_scores[:len(example_images)].argmax(1) == example_labels)
changed &= blur_val_scores[:len(example_images)].argmax(1) != example_labels
examples = torch.where(changed)[0][:4].tolist() or list(range(4))
sharp = pixels(example_images).cpu()
soft = pixels(example_images, blurred=True).cpu()
fig, axes = plt.subplots(2, len(examples), figsize=(3 * len(examples), 5), squeeze=False)
for column, index in enumerate(examples):
    for row, images, scores, name in [(0, sharp, clean_val_scores, 'Clean'), (1, soft, blur_val_scores, 'Blurred')]:
        axis = axes[row, column]
        axis.imshow(images[index, 0], cmap='gray', vmin=0, vmax=1)
        axis.set_title(f'{name}: {CLASS_NAMES[scores[index].argmax()]}\nLabel: {CLASS_NAMES[example_labels[index]]}')
        axis.axis('off')
fig.tight_layout()
plt.show()

In [ ]:
# 04A | Train the Adam comparison and inspect the adaptation errors
@torch.no_grad()
def extract_features(model, batches, blurred=False):
    features, labels = [], []
    for images, targets in batches:
        features.append(model.encoder(pixels(images, blurred)).cpu())
        labels.append(targets)
    return torch.cat(features), torch.cat(labels)

adapt_features, adapt_targets = extract_features(baseline, adapt_loader, blurred=True)
valid_features, valid_targets = extract_features(baseline, val_loader, blurred=True)
head_weight = baseline.classifier.weight.detach().cpu()
head_bias = baseline.classifier.bias.detach().cpu()
adapt_scores = F.linear(adapt_features, head_weight, head_bias)
adapt_cm = confusion(adapt_scores, adapt_targets)
adapt_predictions = adapt_scores.argmax(1)
print('Class IDs:', dict(enumerate(CLASS_NAMES)))
print('Baseline accuracy on blurred adaptation data:', round(accuracy(adapt_scores, adapt_targets), 2))
print('Largest confusions (true -> predicted; count; median wrong-over-true logit margin)')
error_pairs = [(int(adapt_cm[i, j]), i, j) for i in range(10) for j in range(10) if i != j]
for count, actual, predicted in sorted(error_pairs, reverse=True)[:12]:
    mask = (adapt_targets == actual) & (adapt_predictions == predicted)
    margin = (adapt_scores[mask, predicted] - adapt_scores[mask, actual]).median().item()
    print(f'{actual} -> {predicted}: {count}, margin {margin:.2f}')
print('True and predicted class counts:', adapt_cm.sum(1).tolist(), adapt_cm.sum(0).tolist())
print('Biases:', [round(x, 3) for x in head_bias.tolist()])

# Train only the head using the same labeled blurred adaptation set.
adam_repaired = load_weights()
adam_repaired.encoder.requires_grad_(False)
adam_head = adam_repaired.classifier
adam_optimizer = torch.optim.Adam(adam_head.parameters(), lr=0.003)
head_batches = loader(adapt_features, adapt_targets, shuffle=True)
adam_best_score = -1.0
for epoch in range(1, 11):
    for features, labels in head_batches:
        adam_optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(adam_head(features.to(DEVICE)), labels.to(DEVICE))
        loss.backward()
        adam_optimizer.step()
    with torch.no_grad():
        current_acc = accuracy(adam_head(valid_features.to(DEVICE)).cpu(), valid_targets)
    if current_acc > adam_best_score:
        adam_best_score, adam_best_epoch = current_acc, epoch
        save_weights(adam_repaired, RUN_DIR / '04_adam_repaired.pt')
print(f'Adam head-only | best epoch {adam_best_epoch:02d} | blurred validation {adam_best_score:.2f}%')

# Show how often each class is over- or under-predicted at different confidence levels.
for index, name in enumerate(CLASS_NAMES):
    logit = adapt_scores[:, index]
    other = adapt_scores.clone()
    other[:, index] = -torch.inf
    margin = logit - other.max(1).values
    print(f'{index} {name:12s} missed within 0.5: {int(((adapt_targets == index) & (margin < 0) & (margin >= -0.5)).sum()):3d} | '
          f'false positives within 0.5: {int(((adapt_targets != index) & (margin >= 0) & (margin <= 0.5)).sum()):3d}')

In [ ]:
# 04B | Apply the direct LLM patch and run matched random edits
# ChatGPT proposed these offsets after reading the adaptation diagnostics in 04A.
# The six offsets were fixed before viewing patch validation or any test scores.
LLM_PATCH = [
    {'class_id': 0, 'delta': -0.70},  # T-shirt is over-predicted.
    {'class_id': 1, 'delta': +0.45},  # Many missed trousers are close to the decision boundary.
    {'class_id': 4, 'delta': -0.15},  # Coats receive excess pullover and shirt predictions.
    {'class_id': 5, 'delta': +1.25},  # Sandals are often predicted as sneakers or ankle boots.
    {'class_id': 6, 'delta': +0.75},  # The model rarely predicts shirt after blurring.
    {'class_id': 7, 'delta': -0.25},  # Sneakers receive many sandal predictions.
]
RANDOM_SEEDS = range(100, 130)

def patch_bias(original_bias, edits):
    revised = original_bias.clone()
    ids = [edit['class_id'] for edit in edits]
    if len(ids) != len(set(ids)) or any(type(i) is not int or not 0 <= i < len(revised) for i in ids):
        raise ValueError('Each edit must have a unique valid class ID.')
    for edit in edits:
        delta = edit['delta']
        if not math.isfinite(delta) or delta == 0:
            raise ValueError('Each delta must be finite and nonzero.')
        revised[edit['class_id']] += delta
    return revised

llm_repaired = load_weights()
llm_bias = patch_bias(head_bias, LLM_PATCH)
with torch.no_grad():
    llm_repaired.classifier.bias.copy_(llm_bias.to(DEVICE))
assert torch.equal(llm_repaired.classifier.weight.detach().cpu(), head_weight)
assert int((llm_bias != head_bias).sum()) == len(LLM_PATCH)
save_weights(llm_repaired, RUN_DIR / '04_llm_repaired.pt')
patch_record = dict(method='chatgpt_direct_bias_edit', base_sha256=hashlib.sha256(BASELINE_PATH.read_bytes()).hexdigest(),
                    parameter='classifier.bias', edits=[dict(edit, class_name=CLASS_NAMES[edit['class_id']]) for edit in LLM_PATCH])
(RUN_DIR / '04_llm_patch.json').write_text(json.dumps(patch_record, indent=2) + '\n', encoding='utf-8')

# Match the number and absolute sizes of edits in the same tensor.
# Randomize the selected biases, magnitude assignment, and signs.
magnitudes = [abs(edit['delta']) for edit in LLM_PATCH]
random_biases = []
for seed in RANDOM_SEEDS:
    rng = random.Random(seed)
    ids = rng.sample(range(len(CLASS_NAMES)), len(magnitudes))
    edits = [dict(class_id=i, delta=m * rng.choice((-1, 1))) for i, m in zip(ids, magnitudes)]
    random_biases.append((seed, patch_bias(head_bias, edits)))

base_valid = F.linear(valid_features, head_weight, head_bias)
llm_valid = F.linear(valid_features, head_weight, llm_bias)
random_valid = [accuracy(F.linear(valid_features, head_weight, bias), valid_targets) for _, bias in random_biases]
print(f'Blurred validation | baseline {accuracy(base_valid, valid_targets):.2f}% | LLM {accuracy(llm_valid, valid_targets):.2f}% | Adam {adam_best_score:.2f}%')
print(f'Random edits ({len(random_valid)} runs) | mean {statistics.mean(random_valid):.2f}% | SD {statistics.stdev(random_valid):.2f}%')

In [ ]:
# 05 | Evaluate the locked comparison on the official test set and export
# The baseline, Adam checkpoint, direct patch, blur, and control seeds are fixed above.
from IPython.display import FileLink, display

test_data = FashionMNIST(DATA_ROOT, train=False, download=True)
test_loader = loader(*image_set(test_data))
clean_test_features, test_targets = extract_features(baseline, test_loader)
blur_test_features, blur_test_targets = extract_features(baseline, test_loader, blurred=True)
assert torch.equal(test_targets, blur_test_targets)

def read_head(path):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    start = baseline.state_dict()
    assert all(torch.equal(value, start[name].cpu()) for name, value in saved.items() if name.startswith('encoder.'))
    return saved['classifier.weight'], saved['classifier.bias']

def test_result(method, weights, biases, seed=''):
    return dict(method=method, seed=seed,
                clean_accuracy_pct=accuracy(F.linear(clean_test_features, weights, biases), test_targets),
                blurred_accuracy_pct=accuracy(F.linear(blur_test_features, weights, biases), test_targets),
                changed_head_parameters=int((weights != head_weight).sum() + (biases != head_bias).sum()))

rows = [test_result('Original CNN', head_weight, head_bias),
        test_result('Adam head fine-tuning', *read_head(RUN_DIR / '04_adam_repaired.pt')),
        test_result('ChatGPT direct bias edit', *read_head(RUN_DIR / '04_llm_repaired.pt'))]
random_rows = [test_result('Matched random edit', head_weight, bias, seed) for seed, bias in random_biases]
write_csv(RUN_DIR / '05_test_results.csv', rows + random_rows)
print(f'{len(test_targets):,} test images | Gaussian blur sigma={BLUR_SIGMA:g}, kernel={BLUR_KERNEL}')
for row in rows:
    print(f"{row['method']:26s} clean {row['clean_accuracy_pct']:6.2f}% | blurred {row['blurred_accuracy_pct']:6.2f}% | edited {row['changed_head_parameters']:3d}")
clean_control = [row['clean_accuracy_pct'] for row in random_rows]
blur_control = [row['blurred_accuracy_pct'] for row in random_rows]
print(f'Random edits ({len(random_rows)} trials)  clean {statistics.mean(clean_control):.2f}% ± {statistics.stdev(clean_control):.2f} pp | '
      f'blurred {statistics.mean(blur_control):.2f}% ± {statistics.stdev(blur_control):.2f} pp')
base_row, _, llm_row = rows
print(f"Direct edit change | clean {llm_row['clean_accuracy_pct'] - base_row['clean_accuracy_pct']:+.2f} pp | "
      f"blurred {llm_row['blurred_accuracy_pct'] - base_row['blurred_accuracy_pct']:+.2f} pp")
print('Files:')
for path in sorted(RUN_DIR.iterdir()):
    print(f'  {path.name} ({path.stat().st_size:,} bytes)')
archive = shutil.make_archive(str(RUN_DIR) + '_outputs', 'zip', root_dir=RUN_DIR)
display(FileLink(archive, result_html_prefix='Download the checkpoints and results: '))

# In Colab, use this if the link above is unavailable. Run it before ending the runtime.
# from google.colab import files
# files.download(archive)

In [ ]:
# 05B | Download checkpoints and results in Colab
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    display(FileLink(archive))
else:
    import ipywidgets as widgets

    bundle = Path(archive).with_name('CNN_Weight_Repair_Outputs.zip')
    shutil.copyfile(archive, bundle)
    download = widgets.Button(description='Download outputs ZIP', icon='download')
    download.on_click(lambda _: files.download(str(bundle)))
    display(download)